# Shared-memory parallelism

In [12]:
using Pkg
Pkg.instantiate()

In [13]:
Sys.cpu_summary()

Apple M3 Max: 
          speed         user         nice          sys         idle          irq
#1-16  2400 MHz    4701456 s          0 s    2921064 s   71908328 s          0 s


In [14]:
using Hwloc
topology_info()

Machine: 1 (3.13 GB)
 Package: 1 (3.13 GB)
  NUMANode: 1 (3.13 GB)
   L2Cache: 3 (4.0 MB)
    L1Cache: 16 (64.0 kB)
     Core: 16
      PU: 16
       OS_Device: 1


In [15]:
import Base.Threads: @sync, @spawn

In [16]:
function fib(x)
	if x <= 1
		return 1
	else
		b = @spawn fib(x-2)
		a = fib(x-1)
		return a+(fetch(b)::Int)
	end
end

fib (generic function with 1 method)

In [ ]:
fib(10)

## Parallel-loops

In [2]:
import Base.Threads: @threads, nthreads, threadid

In [3]:
let 
	a = zeros(Int, nthreads()*2)
	@threads for i in 1:length(a)
	    a[i] = threadid()
	end
	a
end

2-element Vector{Int64}:
 1
 1

### Schedulers

Julia has diffferent schedulers for parallel for-loops:

- `:dynamic` (the default): Chunks the iteration-space.
- `:greedy`: One-task-per-thread, good for unequal workloads. Iteration-space is interpreted as a channel.
- `:static`: One-task-per-thread, equal division of iteration-space. Can not be nested.


In [5]:
let 
	a = zeros(Int, 1000)
	@threads :greedy for i in 1:length(a)
	    a[i] = threadid()
	end
	a
end

1000-element Vector{Int64}:
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 ⋮
 1
 1
 1
 1
 1
 1
 1
 1
 1

## Parallel Primitives

`@threads` seems nice, but is difficult in many ways. Reductions is an immediate issue:

In [10]:
 function sum_bad(a)
    s = 0
    Threads.@threads for i in a
        s += i
    end
    s
end
sum_bad(1:1_000_000)

500000500000

Instead we have alternatives for things like `map`, `reduce` and `mapreduce` from `OhMyThreads.jl`:

- `tmap`
- `treduce`
- `tmapreduce`
- `tforeach`

In [11]:
using OhMyThreads

In [ ]:
function test_tforeach
	a = zeros(Int, nthreads()*2)
	tforeach(1:length(a)) do i
	    a[i] = threadid()
	end
	a
end
test_tforeach()

In [ ]:
using BenchmarkTools
data = rand(1_000_000 * nthreads());

### Sequential Sum

In [ ]:
function simple_sum(data)
	acc = zero(eltype(data))
	for i in eachindex(data)
		acc += data[i]
	end
	acc
end

In [ ]:
@benchmark sum($data)

In [ ]:
@benchmark simple_sum($data)

### Naive Parallel

In [ ]:
function parallel_sum_falsesharing(data; nchunks = nthreads())
    psums = zeros(eltype(data), nchunks)
    @sync for (c, idcs) in enumerate(OhMyThreads.index_chunks(data; n = nchunks))
        @spawn begin
            for i in idcs
                psums[c] += data[i]
            end
        end
    end
    return sum(psums)
end

In [ ]:
 sum(data) ≈ parallel_sum_falsesharing(data)

In [ ]:
@benchmark parallel_sum_falsesharing($data)

In [ ]:
const CACHE_LINE_SIZE = 64
function parallel_sum_padded(data; nchunks = nthreads())
	# pad each entry
	stride = CACHE_LINE_SIZE ÷ sizeof(eltype(data))
    psums = zeros(eltype(data), nchunks * stride)
    @sync for (c, idcs) in enumerate(OhMyThreads.index_chunks(data; n = nchunks))
        @spawn begin
			c_idx = (c-1) * stride + 1
            for i in idcs
                psums[c_idx] += data[i]
            end
        end
    end
    return sum(psums)
end

In [ ]:
@benchmark parallel_sum_padded($data)

### Task-local parallel sum

In [ ]:
function parallel_sum_tasklocal(data; nchunks = nthreads())
    psums = zeros(eltype(data), nchunks)
    @sync for (c, idcs) in enumerate(OhMyThreads.index_chunks(data; n = nchunks))
        @spawn begin
            local s = zero(eltype(data))
            for i in idcs
                s += data[i]
            end
            psums[c] = s
        end
    end
    return sum(psums)
end

In [ ]:
@benchmark parallel_sum_tasklocal($data)

In [ ]:
function parallel_sum_map(data; nchunks = nthreads())
    psums = zeros(eltype(data), nchunks)
    @sync for (c, idcs) in enumerate(OhMyThreads.index_chunks(data; n = nchunks))
        @spawn begin
            psums[c] = sum(view(data, idcs))
        end
    end
    return sum(psums)
end

In [ ]:
@benchmark parallel_sum_map($data)

In [ ]:
@benchmark treduce($+, $data; ntasks = $nthreads())

## Extra Content

### Channels

Julia tasks are *communicating*, communication can happen with dedicated programming concepts like `Channel`, or directly through memory shared with another tasks.

Channels are first-in-first-out queues that can either be buffered (e.g. contain a reservoir for a number of elements), or un-buffered/blocking. 

In [ ]:
let ch = Channel{Int}(Inf) # buffered
	@sync begin
		for i in 1:10
			@spawn put!(ch, rand(Int))
		end
	end
	close(ch) # Otherwise collect will wait for more data
	collect(ch)
end

### Race-conditions

`Channel` is a concurrent data-structure and ensure that it safe to use with multiple tasks. When we use our own data-structures we have to make sure that we make them safe if necessary, otherwise we will observe data-races.

In [ ]:
mutable struct BrokenCounter
	x::Int
end

In [ ]:
let a = BrokenCounter(0)
	N = 10
	K = 100_000
	@sync for i in 1:N
		@spawn for i in 1:K
			a.x += 1
			GC.safepoint()
		end
	end
	a.x, N*K, a.x/(N*K)
end

### Atomics & Locks

One way this can be fixed is to use atomics. Atomics allow to express the read-and-increment operation as one operation.

In [1]:
mutable struct AtomicCounter
	@atomic x::Int
end

In [ ]:
let a = AtomicCounter(0)
	N = 10
	K = 100_000
	@sync for i in 1:N
		@spawn for i in 1:K
			@atomic a.x += 1
			GC.safepoint()
		end
	end
	a.x, N*K, a.x/(N*K)
end

In [ ]:
let a = AtomicCounter(0)
	a.x = 10
end

In [ ]:
let a = AtomicCounter(0)
	@atomic :sequentially_consistent a.x = 1+2
end

In [ ]:
let a = Base.Lockable(BrokenCounter(0), Base.ReentrantLock())
	N = 10
	K = 100_000
	@sync for i in 1:N
		@spawn for i in 1:K
			@lock(a, a[].x += 1)
			GC.safepoint()
		end
	end
	@lock a begin
		a[].x, N*K, a[].x/(N*K)
	end
end